# Module 10 Solutions: Numerical Methods & Computational Math

This notebook contains complete solutions and working Python code for the Module 10 exercises.

### 1. Floating-Point Machine Epsilon

In [ ]:
import numpy as np

eps = 1.0
while 1.0 + eps / 2.0 != 1.0:
    eps /= 2.0

print("Calculated Machine Epsilon:", eps)
print("NumPy eps:", np.finfo(float).eps)

### 2. FP16 Underflow Demonstration

In [ ]:
x = np.array([1e-5], dtype=np.float16)
print("x in float16:", x[0])
x_squared = x**2
print("x^2 in float16 (underflow):", x_squared[0])  # underflows to 0.0
print("x^2 in float32:", (x.astype(np.float32))**2)

### 3. Condition Number Calculation

In [ ]:
A = np.array([[10, 7], [8, 5.99]])
cond_l1 = np.linalg.cond(A, p=1)
print("L1 Condition Number:", cond_l1)

### 4. Log-Sum-Exp Derivation
$$c + \log \sum_{i=1}^n e^{x_i - c} = c + \log \sum_{i=1}^n \frac{e^{x_i}}{e^c}$$
$$= c + \log \left( \frac{1}{e^c} \sum_{i=1}^n e^{x_i} \right)$$
$$= c + \log(e^{-c}) + \log \sum_{i=1}^n e^{x_i}$$
$$= c - c + \log \sum_{i=1}^n e^{x_i} = \log \sum_{i=1}^n e^{x_i}$$

In [ ]:
# Mathematical proof verified.

### 5. Stable Softmax Implementation

In [ ]:
def stable_softmax(z):
    c = np.max(z)
    e_z = np.exp(z - c)
    return e_z / np.sum(e_z)

print(stable_softmax([1000.0, 1000.0, 1000.0]))
print(stable_softmax([-1000.0, -1000.0, -1000.0]))

### 6. Stable Sigmoid Implementation
For highly negative $x$, $e^{-x}$ overflows. We rewrite:
- For $x \ge 0$: $\sigma(x) = \frac{1}{1 + e^{-x}}$
- For $x < 0$: $\sigma(x) = \frac{e^x}{1 + e^x}$

In [ ]:
def stable_sigmoid(x):
    x = np.array(x, dtype=float)
    return np.where(x >= 0, 1 / (1 + np.exp(-x)), np.exp(x) / (1 + np.exp(x)))

print(stable_sigmoid([1000.0, -1000.0]))

### 7. Simple Linear Congruential Generator

In [ ]:
class LCG:
    def __init__(self, seed=42, a=1664525, c=1013904223, m=2**32):
        self.state = seed
        self.a = a
        self.c = c
        self.m = m
        
    def next(self):
        self.state = (self.a * self.state + self.c) % self.m
        return self.state / self.m

lcg = LCG()
random_numbers = [lcg.next() for _ in range(1000)]
print("First 5 numbers:", random_numbers[:5])

### 8. Power Method for Dominant Eigenvalue

In [ ]:
def power_method(A, num_simulations=100):
    b_k = np.random.rand(A.shape[1])
    for _ in range(num_simulations):
        # calculate the matrix-by-vector product
        b_k1 = np.dot(A, b_k)
        # renormalize the vector
        b_k = b_k1 / np.linalg.norm(b_k1)
    
    # Rayleigh quotient
    eigenvalue = np.dot(b_k, np.dot(A, b_k)) / np.dot(b_k, b_k)
    return eigenvalue, b_k

A = np.array([[4.0, 1.0], [1.0, 3.0]])
eigenval, eigenvector = power_method(A)
print(f"Dominant Eigenvalue: {eigenval:.4f}")
print(f"Eigenvector: {eigenvector}")

### 9. Implement Conjugate Gradient from Scratch

In [ ]:
def conjugate_gradient(A, b, tol=1e-6, max_iter=100):
    x = np.zeros_like(b)
    r = b - A @ x
    p = r.copy()
    rsold = np.dot(r, r)
    
    for i in range(max_iter):
        Ap = A @ p
        alpha = rsold / np.dot(p, Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rsnew = np.dot(r, r)
        if np.sqrt(rsnew) < tol:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x

A = np.array([[4.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, 2.0])
print("CG Solution from scratch:", conjugate_gradient(A, b))

### 10. Monte Carlo Integration for High Dimensions

In [ ]:
# Volume of 10-dimensional hypersphere: points inside [-1, 1]^10 with L2 norm <= 1
# Analytical volume of 10D hypersphere is pi^5 / 5! = (pi^5) / 120 = 2.550164
for N in [10**4, 10**5, 10**6]:
    pts = np.random.uniform(-1, 1, (N, 10))
    inside = np.sum(pts**2, axis=1) <= 1.0
    fraction = np.sum(inside) / N
    # Volume of bounding box is 2^10 = 1024
    volume_est = fraction * 1024
    print(f"N = {N}: Estimated Volume = {volume_est:.4f} (True: 2.5502)")

### 11. Importance Sampling

In [ ]:
import scipy.stats as stats

N = 100000
# Sample from proposal q
q_samples = np.random.normal(0, np.sqrt(2), N)

# Compute weight w(x) = p(x) / q(x)
p_density = stats.norm.pdf(q_samples, loc=2, scale=1)
q_density = stats.norm.pdf(q_samples, loc=0, scale=np.sqrt(2))
weights = p_density / q_density

# estimate E[x^2] for x~p
# True value is Var(x) + E[x]^2 = 1 + 2^2 = 5
est = np.mean(q_samples**2 * weights)
print(f"Importance sampling estimate: {est:.4f} (True: 5.0)")

### 12. Cholesky Factorization for Jitter
Due to roundoff error, a theoretical covariance matrix $K$ might have tiny negative eigenvalues, causing Cholesky factorization to fail. 
Adding a small positive value $\epsilon$ to the diagonal (known as **jitter**, typically $\epsilon = 10^{-6} \times I$) shifts all eigenvalues up by $\epsilon$, restoring positive definiteness:
$$\lambda_i(K + \epsilon I) = \lambda_i(K) + \epsilon > 0$$

In [ ]:
K = np.array([[1.0, 1.0], [1.0, 1.0]])  # rank 1, positive semi-definite
try:
    np.linalg.cholesky(K)
except np.linalg.LinAlgError as e:
    print("Cholesky failed as expected!", e)
    
K_jitter = K + 1e-8 * np.eye(2)
L = np.linalg.cholesky(K_jitter)
print("Cholesky succeeded with jitter:\n", L)

### 13. Euler's Method vs Runge-Kutta

In [ ]:
dt = 0.4
t_end = 2.0
steps = int(t_end / dt)

# Euler
y_euler = 1.0
for _ in range(steps):
    y_euler = y_euler + dt * (-2 * y_euler)

# RK4
y_rk4 = 1.0
for _ in range(steps):
    k1 = -2 * y_rk4
    k2 = -2 * (y_rk4 + 0.5 * dt * k1)
    k3 = -2 * (y_rk4 + 0.5 * dt * k2)
    k4 = -2 * (y_rk4 + dt * k3)
    y_rk4 = y_rk4 + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

print(f"True: {np.exp(-4):.6f}, Euler: {y_euler:.6f}, RK4: {y_rk4:.6f}")

### 14. Truncated SVD for Low-Rank Approximation

In [ ]:
A = np.zeros((100, 100))
for i in range(100):
    for j in range(100):
        A[i, j] = np.sin(i / 10.0) + np.cos(j / 10.0)

U, S, Vt = np.linalg.svd(A)
# Rank 2 approximation
A_approx = U[:, :2] @ np.diag(S[:2]) @ Vt[:2, :]
error = np.linalg.norm(A - A_approx)
print(f"Reconstruction Error with rank-2 approximation: {error:.6e}")

### 15. Newton-Raphson for Logistic Regression MLE

In [ ]:
X = np.array([[1.0, 0.5], [1.0, 1.5], [1.0, 2.5], [1.0, 3.5]])
y = np.array([0.0, 0.0, 1.0, 1.0])

w = np.array([0.0, 0.0])
for _ in range(10):
    p = 1 / (1 + np.exp(-X @ w))
    gradient = X.T @ (p - y)
    W_diag = p * (1 - p)
    Hessian = X.T @ np.diag(W_diag) @ X
    w -= np.linalg.solve(Hessian, gradient)

print("Fitted Weights (intercept, slope):", w)